In [5]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
#settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/edvinramstrom/Documents/KTH/ID2223 Scalable/mlfs-book


<span style="font-width:bold; font-size: 3rem; color:#333;">- Part 02: Daily Feature Pipeline for Air Quality (aqicn.org) and weather (openmeteo)</span>

## 🗒️ This notebook is divided into the following sections:
1. Download and Parse Data
2. Feature Group Insertion


__This notebook should be scheduled to run daily__

In the book, we use a GitHub Action stored here:
[.github/workflows/air-quality-daily.yml](https://github.com/featurestorebook/mlfs-book/blob/main/.github/workflows/air-quality-daily.yml)

However, you are free to use any Python Orchestration tool to schedule this program to run daily.

### <span style='color:#ff5f27'> 📝 Imports

In [6]:
import datetime
import time
import requests
import pandas as pd
import hopsworks
from mlfs.airquality import util
from mlfs import config
import json
import os
import warnings
warnings.filterwarnings("ignore")

### <span style='color:#ff5f27'> 🎯 Sensor Configuration</span>

**Change the sensor street name below to select which sensor to process:**

In [7]:
# === SENSOR SELECTION - SET THIS PARAMETER ===
# This can be set via papermill: papermill input.ipynb output.ipynb -p SENSOR_STREET "arnulf-klett-platz"
SENSOR_STREET = "am-neckartor"  # Options: "arnulf-klett-platz", "am-neckartor"

print(f"Processing sensor: {SENSOR_STREET}")

Processing sensor: am-neckartor


## <span style='color:#ff5f27'> 🌍 Get the Sensor URL, Country, City, Street names from Hopsworks </span>

__Update the values in the cell below.__

__These should be the same values as in notebook 1 - the feature backfill notebook__


In [8]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store() 
secrets = hopsworks.get_secrets_api()

# This line will fail if you have not registered the AQICN_API_KEY as a secret in Hopsworks
AQICN_API_KEY = secrets.get_secret("AQICN_API_KEY").value

# Load sensor-specific configuration from Hopsworks secret
secret_name = f"SENSOR_LOCATION_JSON_{SENSOR_STREET.upper().replace('-', '_')}"
location_str = secrets.get_secret(secret_name).value
location = json.loads(location_str)

country=location['country']
city=location['city']
street=location['street']
aqicn_url=location['aqicn_url']
latitude=location['latitude']
longitude=location['longitude']

today = datetime.date.today()

print(f"Loaded configuration for sensor: {street}")
print(f"Secret name: {secret_name}")

location_str

2025-11-18 17:04:06,971 INFO: Closing external client and cleaning up certificates.
Connection closed.
2025-11-18 17:04:06,974 INFO: Initializing external client
2025-11-18 17:04:06,974 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-11-18 17:04:08,471 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1298581
Loaded configuration for sensor: am-neckartor
Secret name: SENSOR_LOCATION_JSON_AM_NECKARTOR


'{"country": "germany", "city": "stuttgart", "street": "am-neckartor", "aqicn_url": "https://api.waqi.info/feed/@11228", "latitude": "48.7997", "longitude": "9.1994"}'

### <span style="color:#ff5f27;"> 🔮 Get references to the Feature Groups </span>

In [9]:
# Retrieve feature groups
air_quality_fg = fs.get_feature_group(
    name='air_quality',
    version=3,
)
weather_fg = fs.get_feature_group(
    name='weather',
    version=2,
)

---

## <span style='color:#ff5f27'> 🌫 Retrieve Today's Air Quality data (PM2.5) from the AQI API</span>


In [10]:
import requests
import pandas as pd

aq_today_df = util.get_pm25(aqicn_url, country, city, street, today, AQICN_API_KEY)

lag_features = util.get_pm25_lagged_features(air_quality_fg, today, country, city, street)
for lag_name, lag_value in lag_features.items():
    aq_today_df[lag_name] = lag_value

# add day of week feature
aq_today_df['day_of_week'] = today.weekday()
# make int32
aq_today_df['day_of_week'] = aq_today_df['day_of_week'].astype('int32')

aq_today_df


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.13s) 


,pm25,country,city,street,date,url,pm25_lag1,pm25_lag2,pm25_lag3,day_of_week
0,27.0,germany,stuttgart,am-neckartor,2025-11-18,https://api.waqi.info/feed/@11228,None,None,None,1


In [11]:
aq_today_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   pm25         1 non-null      float32       
 1   country      1 non-null      object        
 2   city         1 non-null      object        
 3   street       1 non-null      object        
 4   date         1 non-null      datetime64[ns]
 5   url          1 non-null      object        
 6   pm25_lag1    0 non-null      object        
 7   pm25_lag2    0 non-null      object        
 8   pm25_lag3    0 non-null      object        
 9   day_of_week  1 non-null      int32         
dtypes: datetime64[ns](1), float32(1), int32(1), object(7)
memory usage: 200.0+ bytes


## <span style='color:#ff5f27'> 🌦 Get Weather Forecast data</span>

In [12]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city
daily_df

Coordinates 48.75°N 9.25°E
Elevation 248.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,relative_humidity_2m_mean,surface_pressure_mean,city
0,2025-11-18,4.40,0.0,3.075841,159.443878,70.0,990.585815,stuttgart
1,2025-11-19,6.50,0.0,13.493999,189.210953,57.0,978.392761,stuttgart
2,2025-11-20,2.65,0.2,11.720751,280.619598,82.0,981.181702,stuttgart
3,2025-11-21,2.40,0.1,11.620809,16.189287,68.0,988.137146,stuttgart
4,2025-11-22,-1.75,0.0,4.394360,55.007900,82.0,995.042542,stuttgart
5,2025-11-23,1.30,0.0,12.413476,106.858482,47.0,980.355469,stuttgart
6,2025-11-24,0.85,0.1,1.080000,90.000000,79.0,977.009277,stuttgart


In [13]:
daily_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         7 non-null      datetime64[ns]
 1   temperature_2m_mean          7 non-null      float32       
 2   precipitation_sum            7 non-null      float32       
 3   wind_speed_10m_max           7 non-null      float32       
 4   wind_direction_10m_dominant  7 non-null      float32       
 5   relative_humidity_2m_mean    7 non-null      float32       
 6   surface_pressure_mean        7 non-null      float32       
 7   city                         7 non-null      object        
dtypes: datetime64[ns](1), float32(6), object(1)
memory usage: 408.0+ bytes


## <span style="color:#ff5f27;">⬆️ Uploading new data to the Feature Store</span>

In [14]:
# Insert new data
air_quality_fg.insert(aq_today_df)


2025-11-18 17:04:16,013 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1298581/fs/1286212/fg/1711467


Uploading Dataframe: 100.00% |██████████| Rows 1/1 | Elapsed Time: 00:01 | Remaining Time: 00:00


(Job('air_quality_3_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "pm25",
           "min_value": -0.1,
           "max_value": 500.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 752664
         }
       },
       "result": {
         "observed_value": 27.0,
         "element_count": 1,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-11-18T04:04:16.000013Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     }
   ],
   "evaluation_parameters": {},
   "statistics": {
     "evaluated_expectations": 1,
     "successful_expectat

In [15]:
# Insert new data
weather_fg.insert(daily_df, wait=True)

2025-11-18 17:04:24,950 INFO: 	2 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1298581/fs/1286212/fg/1703359


KeyboardInterrupt: 

## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Training Pipeline
 </span> 

In the following notebook you will read from a feature group and create training dataset within the feature store
